In [1]:
# Install dependencies (run once in Colab)
!pip -q install transformers datasets torch scikit-learn pandas matplotlib seaborn accelerate sentencepiece


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import json
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)


In [4]:
# =========================
# CONFIG
# =========================
DATA_PATH = "/content/drive/MyDrive/conversation_en.json"

# 5-class original augmentation pool. Do NOT use the 4-class-specific file here.
AUGMENTED_TRAIN_PATH = "/content/drive/MyDrive/final_train_augmented_v2.csv"

TRANSLATED_CN_PATH = "/content/drive/MyDrive/conversation_cn_translated_en_deepseek_v4flash.json"
USE_TRANSLATED_CN_AS_EXTRA_TRAIN = True

RESULTS_DIR = "/content/drive/MyDrive/intent9Aug/5classFinalscript_noleak_fullaugment_cn"
os.makedirs(RESULTS_DIR, exist_ok=True)

MAX_LEN = 256
RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 5

VALID_LABELS = ["Chitchat", "Interpret", "Reveal", "Revise", "Inquire"]
label2id = {"Chitchat": 0, "Interpret": 1, "Reveal": 2, "Revise": 3, "Inquire": 4}
id2label = {v: k for k, v in label2id.items()}

print("Experiment: clean 5-class single split, full LLM augmentation pool + translated CN")
print("Labels:", label2id)
print("LR:", LEARNING_RATE, "weight_decay:", WEIGHT_DECAY)
print("Results:", RESULTS_DIR)


Experiment: clean 5-class single split, full LLM augmentation pool + translated CN
Labels: {'Chitchat': 0, 'Interpret': 1, 'Reveal': 2, 'Revise': 3, 'Inquire': 4}
LR: 2e-05 weight_decay: 0.01
Results: /content/drive/MyDrive/intent9Aug/5classFinalscript_noleak_fullaugment_cn


In [5]:
# =========================
# LOAD ORIGINAL ENGLISH DATA
# =========================
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

all_rows = []
for conv in raw:
    conv_id = str(conv.get("conv_id"))
    for i, t in enumerate(conv.get("conversation", [])):
        all_rows.append({
            "conversation_id": conv_id,
            "turn_index": int(i),
            "role": str(t.get("role", "")).lower(),
            "text": t.get("content") or "",
            "intent": t.get("intent"),
        })

df_turns_full = pd.DataFrame(all_rows).sort_values(
    ["conversation_id", "turn_index"]
).reset_index(drop=True)

df_turns_full["conversation_id"] = df_turns_full["conversation_id"].astype(str)
df_turns_full["turn_index"] = df_turns_full["turn_index"].astype(int)
df_turns_full["role"] = df_turns_full["role"].astype(str).str.lower()
df_turns_full["text"] = df_turns_full["text"].fillna("").astype(str)

target_df = df_turns_full[
    (df_turns_full["role"] == "user")
    & (df_turns_full["intent"].notna())
    & (df_turns_full["intent"].isin(VALID_LABELS))
].copy().reset_index(drop=True)

turns_by_conv_full = {
    cid: g.sort_values("turn_index").reset_index(drop=True)
    for cid, g in df_turns_full.groupby("conversation_id")
}

print("Original full turn rows:", len(df_turns_full))
print("Original 5-class target rows:", len(target_df))
print("Original target conversations:", target_df["conversation_id"].nunique())
print("Original 5-class distribution:")
print(target_df["intent"].value_counts())
print(target_df[["conversation_id", "turn_index", "role", "intent"]].head())


Original full turn rows: 5633
Original 5-class target rows: 3196
Original target conversations: 826
Original 5-class distribution:
intent
Interpret    1813
Reveal       1031
Chitchat      268
Revise         67
Inquire        17
Name: count, dtype: int64
  conversation_id  turn_index  role     intent
0          100204           0  user     Reveal
1          100204           2  user     Reveal
2          100325           0  user   Chitchat
3          100325           2  user     Reveal
4          100325           4  user  Interpret


In [6]:
# =========================
# GROUPED SINGLE SPLIT — ORIGINAL ENGLISH ONLY
# =========================
gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(
    gss1.split(target_df, groups=target_df["conversation_id"].astype(str))
)

train_val_df = target_df.iloc[train_val_idx].reset_index(drop=True)
test_df = target_df.iloc[test_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE_WITHIN_TRAIN, random_state=RANDOM_STATE)
train_idx, val_idx = next(
    gss2.split(train_val_df, groups=train_val_df["conversation_id"].astype(str))
)

train_df_original = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

train_cids = set(train_df_original["conversation_id"].astype(str))
val_cids = set(val_df["conversation_id"].astype(str))
test_cids = set(test_df["conversation_id"].astype(str))

print("Original train rows:", len(train_df_original))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nConversation counts:")
print("Train conversations:", train_df_original["conversation_id"].nunique())
print("Val conversations:", val_df["conversation_id"].nunique())
print("Test conversations:", test_df["conversation_id"].nunique())

print("\nOriginal split conversation overlap:")
print("Train ∩ Val:", len(train_cids & val_cids))
print("Train ∩ Test:", len(train_cids & test_cids))
print("Val ∩ Test:", len(val_cids & test_cids))

assert train_cids.isdisjoint(val_cids)
assert train_cids.isdisjoint(test_cids)
assert val_cids.isdisjoint(test_cids)

print("\nTrain distribution:")
print(train_df_original["intent"].value_counts())
print("\nVal distribution:")
print(val_df["intent"].value_counts())
print("\nTest distribution:")
print(test_df["intent"].value_counts())


Original train rows: 2022
Validation rows: 522
Test rows: 652

Conversation counts:
Train conversations: 528
Val conversations: 132
Test conversations: 166

Original split conversation overlap:
Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0

Train distribution:
intent
Interpret    1131
Reveal        670
Chitchat      168
Revise         45
Inquire         8
Name: count, dtype: int64

Val distribution:
intent
Interpret    304
Reveal       168
Chitchat      36
Revise        10
Inquire        4
Name: count, dtype: int64

Test distribution:
intent
Interpret    378
Reveal       193
Chitchat      64
Revise        12
Inquire        5
Name: count, dtype: int64


In [7]:
# =========================
# LLM AUGMENTATION — TRAIN-SOURCE ONLY
# =========================
aug_all_df = pd.read_csv(AUGMENTED_TRAIN_PATH)

required_aug_columns = {"conversation_id", "turn_index", "role", "text", "intent", "is_augmented"}
missing_columns = required_aug_columns - set(aug_all_df.columns)
assert not missing_columns, f"Missing augmentation columns: {missing_columns}"

aug_all_df["conversation_id"] = aug_all_df["conversation_id"].astype(str)
aug_all_df["turn_index"] = pd.to_numeric(aug_all_df["turn_index"], errors="raise").astype(int)
aug_all_df["role"] = aug_all_df["role"].fillna("").astype(str).str.strip().str.lower()
aug_all_df["text"] = aug_all_df["text"].fillna("").astype(str)
aug_all_df["intent"] = aug_all_df["intent"].fillna("").astype(str).str.strip()

is_genuine_augmentation = (
    aug_all_df["is_augmented"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "1.0"])
)

aug_genuine_df = aug_all_df[
    is_genuine_augmentation
    & (aug_all_df["role"] == "user")
    & (aug_all_df["intent"].isin(VALID_LABELS))
].copy().reset_index(drop=True)

assert len(aug_genuine_df) > 0, "No genuine 5-class augmented rows found."

aug_train_df = aug_genuine_df[
    aug_genuine_df["conversation_id"].astype(str).isin(train_cids)
].copy().reset_index(drop=True)

aug_cids = set(aug_train_df["conversation_id"].astype(str))

print("Rows in augmentation CSV:", len(aug_all_df))
print("Original/non-augmented rows ignored:", int((~is_genuine_augmentation).sum()))
print("Genuine valid 5-class augmentation rows:", len(aug_genuine_df))
print("Train-source genuine augmentation kept:", len(aug_train_df))
print("Genuine augmentation rows excluded because source was not TRAIN:", len(aug_genuine_df) - len(aug_train_df))
print("\nGenuine augmentation pool distribution:")
print(aug_genuine_df["intent"].value_counts())
print("\nTrain-source augmentation distribution:")
print(aug_train_df["intent"].value_counts())

print("\nAugmentation source safety checks:")
print("Aug source ∩ Val:", len(aug_cids & val_cids))
print("Aug source ∩ Test:", len(aug_cids & test_cids))
print("Aug source outside Train:", len(aug_cids - train_cids))

assert aug_cids.issubset(train_cids), "Unexpected augmentation source outside original train split."
assert aug_cids.isdisjoint(val_cids), "LEAKAGE: validation-source augmentation entered training."
assert aug_cids.isdisjoint(test_cids), "LEAKAGE: test-source augmentation entered training."

# Verify every kept synthetic row maps to an original TRAIN source turn and preserves label.
source_5class = train_df_original[["conversation_id", "turn_index", "intent"]].copy()
source_5class["conversation_id"] = source_5class["conversation_id"].astype(str)
source_5class["turn_index"] = source_5class["turn_index"].astype(int)

aug_source_check = aug_train_df.merge(
    source_5class,
    on=["conversation_id", "turn_index"],
    how="left",
    suffixes=("_aug", "_source"),
)

missing_source_turns = int(aug_source_check["intent_source"].isna().sum())
changed_labels = int((
    aug_source_check["intent_source"].notna()
    & (aug_source_check["intent_aug"] != aug_source_check["intent_source"])
).sum())

print("Augmented rows missing original TRAIN source turn:", missing_source_turns)
print("Augmented rows with changed labels:", changed_labels)

assert missing_source_turns == 0
assert changed_labels == 0


Rows in augmentation CSV: 2296
Original/non-augmented rows ignored: 2022
Genuine valid 5-class augmentation rows: 274
Train-source genuine augmentation kept: 274
Genuine augmentation rows excluded because source was not TRAIN: 0

Genuine augmentation pool distribution:
intent
Chitchat    100
Revise      100
Inquire      74
Name: count, dtype: int64

Train-source augmentation distribution:
intent
Chitchat    100
Revise      100
Inquire      74
Name: count, dtype: int64

Augmentation source safety checks:
Aug source ∩ Val: 0
Aug source ∩ Test: 0
Aug source outside Train: 0
Augmented rows missing original TRAIN source turn: 0
Augmented rows with changed labels: 0


In [8]:
# =========================
# LOAD TRANSLATED CN — INDEPENDENT EXTERNAL TRAINING CORPUS
# =========================
cn_train_df = pd.DataFrame()
cn_turns_full = pd.DataFrame()

if USE_TRANSLATED_CN_AS_EXTRA_TRAIN:
    with open(TRANSLATED_CN_PATH, "r", encoding="utf-8") as f:
        cn_raw = json.load(f)

    cn_rows = []
    for conv in cn_raw:
        cid = "CN_" + str(conv.get("conv_id"))
        for i, t in enumerate(conv.get("conversation", [])):
            cn_rows.append({
                "conversation_id": cid,
                "turn_index": int(i),
                "role": str(t.get("role", "")).lower(),
                "text": t.get("content") or "",
                "intent": t.get("intent"),
            })

    cn_turns_full = pd.DataFrame(cn_rows).sort_values(
        ["conversation_id", "turn_index"]
    ).reset_index(drop=True)
    cn_turns_full["conversation_id"] = cn_turns_full["conversation_id"].astype(str)
    cn_turns_full["turn_index"] = cn_turns_full["turn_index"].astype(int)
    cn_turns_full["role"] = cn_turns_full["role"].astype(str).str.lower()
    cn_turns_full["text"] = cn_turns_full["text"].fillna("").astype(str)

    cn_train_df = cn_turns_full[
        (cn_turns_full["role"] == "user")
        & (cn_turns_full["intent"].notna())
        & (cn_turns_full["intent"].isin(VALID_LABELS))
    ].copy().reset_index(drop=True)

    cn_turns_by_conv_full = {
        cid: g.sort_values("turn_index").reset_index(drop=True)
        for cid, g in cn_turns_full.groupby("conversation_id")
    }
    turns_by_conv_full.update(cn_turns_by_conv_full)

cn_cids = set(cn_train_df["conversation_id"].astype(str)) if len(cn_train_df) else set()

print("Translated CN 5-class rows retained:", len(cn_train_df))
if len(cn_train_df):
    print(cn_train_df["intent"].value_counts())

print("\nCN namespace safety checks:")
print("CN ∩ English Train:", len(cn_cids & train_cids))
print("CN ∩ English Val:", len(cn_cids & val_cids))
print("CN ∩ English Test:", len(cn_cids & test_cids))

assert cn_cids.isdisjoint(train_cids)
assert cn_cids.isdisjoint(val_cids)
assert cn_cids.isdisjoint(test_cids)


Translated CN 5-class rows retained: 3056
intent
Interpret    1590
Reveal        975
Chitchat      275
Revise        215
Inquire         1
Name: count, dtype: int64

CN namespace safety checks:
CN ∩ English Train: 0
CN ∩ English Val: 0
CN ∩ English Test: 0


In [9]:
# =========================
# FINAL TRAIN + LEAKAGE AUDIT
# =========================
train_df = pd.concat(
    [train_df_original, aug_train_df, cn_train_df],
    ignore_index=True,
).reset_index(drop=True)

print("Final train rows:", len(train_df))
print("  original English:", len(train_df_original))
print("  genuine safe augmentation:", len(aug_train_df))
print("  independent translated CN:", len(cn_train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nFinal train label distribution:")
print(train_df["intent"].value_counts())
print("\nValidation label distribution:")
print(val_df["intent"].value_counts())
print("\nTest label distribution:")
print(test_df["intent"].value_counts())

missing_context_cids = set(train_df["conversation_id"].astype(str)) - set(turns_by_conv_full.keys())
print("Missing context conversation IDs:", len(missing_context_cids))
assert len(missing_context_cids) == 0

# Exact text overlap is diagnostic only; conversation-source isolation is the leakage rule.
def norm_text(x):
    x = str(x).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

train_texts = set(train_df["text"].map(norm_text)) - {""}
val_texts = set(val_df["text"].map(norm_text)) - {""}
test_texts = set(test_df["text"].map(norm_text)) - {""}
print("\nExact target-text overlap train/val (diagnostic):", len(train_texts & val_texts))
print("Exact target-text overlap train/test (diagnostic):", len(train_texts & test_texts))

audit = pd.DataFrame([{
    "original_train_rows": len(train_df_original),
    "augmentation_csv_rows": len(aug_all_df),
    "genuine_5class_aug_rows": len(aug_genuine_df),
    "safe_train_source_aug_rows": len(aug_train_df),
    "genuine_aug_rows_excluded_nontrain_source": len(aug_genuine_df) - len(aug_train_df),
    "independent_cn_rows": len(cn_train_df),
    "final_train_rows": len(train_df),
    "val_rows": len(val_df),
    "test_rows": len(test_df),
    "train_val_cid_overlap": len(train_cids & val_cids),
    "train_test_cid_overlap": len(train_cids & test_cids),
    "val_test_cid_overlap": len(val_cids & test_cids),
    "aug_val_source_overlap": len(aug_cids & val_cids),
    "aug_test_source_overlap": len(aug_cids & test_cids),
    "aug_missing_source_turns": missing_source_turns,
    "aug_changed_labels": changed_labels,
    "cn_train_cid_overlap": len(cn_cids & train_cids),
    "cn_val_cid_overlap": len(cn_cids & val_cids),
    "cn_test_cid_overlap": len(cn_cids & test_cids),
}])

audit.to_csv(os.path.join(RESULTS_DIR, "leakage_audit.csv"), index=False)
display(audit)

assert (audit[[
    "train_val_cid_overlap",
    "train_test_cid_overlap",
    "val_test_cid_overlap",
    "aug_val_source_overlap",
    "aug_test_source_overlap",
    "aug_missing_source_turns",
    "aug_changed_labels",
    "cn_train_cid_overlap",
    "cn_val_cid_overlap",
    "cn_test_cid_overlap",
]] == 0).all().all(), "Leakage audit failed."

print("Leakage audit passed.")


Final train rows: 5352
  original English: 2022
  genuine safe augmentation: 274
  independent translated CN: 3056
Validation rows: 522
Test rows: 652

Final train label distribution:
intent
Interpret    2721
Reveal       1645
Chitchat      543
Revise        360
Inquire        83
Name: count, dtype: int64

Validation label distribution:
intent
Interpret    304
Reveal       168
Chitchat      36
Revise        10
Inquire        4
Name: count, dtype: int64

Test label distribution:
intent
Interpret    378
Reveal       193
Chitchat      64
Revise        12
Inquire        5
Name: count, dtype: int64
Missing context conversation IDs: 0

Exact target-text overlap train/val (diagnostic): 55
Exact target-text overlap train/test (diagnostic): 64


,original_train_rows,augmentation_csv_rows,genuine_5class_aug_rows,safe_train_source_aug_rows,genuine_aug_rows_excluded_nontrain_source,independent_cn_rows,final_train_rows,val_rows,test_rows,train_val_cid_overlap,train_test_cid_overlap,val_test_cid_overlap,aug_val_source_overlap,aug_test_source_overlap,aug_missing_source_turns,aug_changed_labels,cn_train_cid_overlap,cn_val_cid_overlap,cn_test_cid_overlap
0,2022,2296,274,274,0,3056,5352,522,652,0,0,0,0,0,0,0,0,0,0


Leakage audit passed.


In [10]:
# =========================
# CONTEXT BUILDER
# =========================
def build_context_dataset(df_split, turns_by_conv):
    X, y = [], []
    for _, row in df_split.iterrows():
        cid = str(row["conversation_id"])
        ti = int(row["turn_index"])
        conv_turns = turns_by_conv[cid]
        prev_turns = conv_turns[conv_turns["turn_index"] < ti]

        history = []
        for _, p in prev_turns.iterrows():
            role = str(p["role"]).upper().strip()
            text = str(p["text"]).strip()
            if text:
                history.append(f"{role}: {text}")

        current_turn = str(row["text"]).strip()
        full_text = ("\n".join(history) + "\n" if history else "") + f"USER: {current_turn}"
        X.append(full_text)
        y.append(row["intent"])
    return np.array(X), np.array(y)

X_train, y_train = build_context_dataset(train_df, turns_by_conv_full)
X_val, y_val = build_context_dataset(val_df, turns_by_conv_full)
X_test, y_test = build_context_dataset(test_df, turns_by_conv_full)

print("Context-aware train:", len(X_train))
print("Context-aware val:", len(X_val))
print("Context-aware test:", len(X_test))

print("\nTrain distribution:")
print(pd.Series(y_train).value_counts())
print("\nVal distribution:")
print(pd.Series(y_val).value_counts())
print("\nTest distribution:")
print(pd.Series(y_test).value_counts())

revise_indices = [i for i, l in enumerate(y_train) if l == "Revise"]
if revise_indices:
    print("\n--- Sample Revise context ---")
    for line in X_train[revise_indices[0]].split("\n"):
        print(" ", line[:110])


Context-aware train: 5352
Context-aware val: 522
Context-aware test: 652

Train distribution:
Interpret    2721
Reveal       1645
Chitchat      543
Revise        360
Inquire        83
Name: count, dtype: int64

Val distribution:
Interpret    304
Reveal       168
Chitchat      36
Revise        10
Inquire        4
Name: count, dtype: int64

Test distribution:
Interpret    378
Reveal       193
Chitchat      64
Revise        12
Inquire        5
Name: count, dtype: int64

--- Sample Revise context ---
  USER: hello，I want buy a mug
  SYSTEM: how about these？
  USER: ok,the first one i will choose.


In [11]:
# =========================
# LABEL IDS + DISTRIBUTION + TRAIN-ONLY CLASS WEIGHTS
# =========================
y_train_ids = np.array([label2id[x] for x in y_train], dtype=np.int64)
y_val_ids = np.array([label2id[x] for x in y_val], dtype=np.int64)
y_test_ids = np.array([label2id[x] for x in y_test], dtype=np.int64)

train_dist = pd.Series(y_train).value_counts(normalize=True)
val_dist = pd.Series(y_val).value_counts(normalize=True)
test_dist = pd.Series(y_test).value_counts(normalize=True)

dist_df = pd.DataFrame({
    "train_%": train_dist,
    "val_%": val_dist,
    "test_%": test_dist,
}).reindex(VALID_LABELS).fillna(0)

print("Label distribution (%):")
print(dist_df)

classes = np.arange(len(VALID_LABELS))
missing = set(classes) - set(np.unique(y_train_ids))
assert not missing, f"Missing train classes: {[id2label[i] for i in sorted(missing)]}"

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_ids,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("\nClass weights:")
for cls_id, w in zip(classes, class_weights):
    print(f"  {id2label[cls_id]}: {w:.4f}")


Label distribution (%):
            train_%     val_%    test_%
Chitchat   0.101457  0.068966  0.098160
Interpret  0.508408  0.582375  0.579755
Reveal     0.307362  0.321839  0.296012
Revise     0.067265  0.019157  0.018405
Inquire    0.015508  0.007663  0.007669

Class weights:
  Chitchat: 1.9713
  Interpret: 0.3934
  Reveal: 0.6507
  Revise: 2.9733
  Inquire: 12.8964


In [14]:
# =========================
# TRAINER + METRICS + HELPERS
# =========================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        loss = nn.CrossEntropyLoss(weight=weight)(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        labels=np.arange(len(VALID_LABELS)),
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": p,
        "macro_recall": r,
        "macro_f1": f1,
    }


def make_hf_datasets(tokenizer, X_tr, y_tr_ids, X_va, y_va_ids, X_te, y_te_ids):
    tr_enc = tokenizer(list(X_tr), truncation=True, padding=True, max_length=MAX_LEN)
    va_enc = tokenizer(list(X_va), truncation=True, padding=True, max_length=MAX_LEN)
    te_enc = tokenizer(list(X_te), truncation=True, padding=True, max_length=MAX_LEN)
    return (
        Dataset.from_dict({"input_ids": tr_enc["input_ids"], "attention_mask": tr_enc["attention_mask"], "labels": y_tr_ids.tolist()}),
        Dataset.from_dict({"input_ids": va_enc["input_ids"], "attention_mask": va_enc["attention_mask"], "labels": y_va_ids.tolist()}),
        Dataset.from_dict({"input_ids": te_enc["input_ids"], "attention_mask": te_enc["attention_mask"], "labels": y_te_ids.tolist()}),
    )


def evaluate_and_report(trainer, dataset, y_ids, X_texts, split_name, model_name):
    pred_out = trainer.predict(dataset)
    y_pred_ids = np.argmax(pred_out.predictions, axis=1)

    y_pred = [id2label[i] for i in y_pred_ids]
    y_true = [id2label[i] for i in y_ids]

    report = classification_report(
        y_true,
        y_pred,
        labels=VALID_LABELS,
        digits=3,
        zero_division=0,
    )
    print(f"=== {model_name} — {split_name} CLASSIFICATION REPORT ===")
    print(report)

    cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS)
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{l}" for l in VALID_LABELS],
        columns=[f"pred_{l}" for l in VALID_LABELS],
    )
    print(f"=== {model_name} — {split_name} CONFUSION MATRIX ===")
    display(cm_df)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=VALID_LABELS, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=VALID_LABELS, average="weighted", zero_division=0
    )
    _, _, per_class_f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=VALID_LABELS, average=None, zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    metrics = {
        "split": split_name.upper(),
        "model": model_name,
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1,
        "samples": len(y_true),
    }
    for label, f1_val, supp in zip(VALID_LABELS, per_class_f1, support):
        metrics[f"{label.lower()}_f1"] = f1_val
        metrics[f"{label.lower()}_support"] = supp

    tag = f"{model_name.lower().replace('/', '_')}_{split_name.lower()}"
    pred_df = pd.DataFrame({"text": X_texts, "true_label": y_true, "predicted_label": y_pred})
    pred_df.to_csv(os.path.join(RESULTS_DIR, f"{tag}_predictions.csv"), index=False)
    cm_df.to_csv(os.path.join(RESULTS_DIR, f"{tag}_cm.csv"))

    errors = pred_df[pred_df["true_label"] != pred_df["predicted_label"]]
    confused_pairs = (
        errors[["true_label", "predicted_label"]]
        .value_counts()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    if split_name.upper() in ["VAL", "TEST"]:
        print(f"=== {model_name} {split_name} TOP CONFUSED PAIRS ===")
        display(confused_pairs.head(15))
        errors.to_csv(os.path.join(RESULTS_DIR, f"{tag}_misclassified.csv"), index=False)
        confused_pairs.to_csv(os.path.join(RESULTS_DIR, f"{tag}_confused_pairs.csv"), index=False)

    with open(os.path.join(RESULTS_DIR, f"{tag}_report.txt"), "w", encoding="utf-8") as f:
        f.write(f"=== {model_name} — {split_name} CLASSIFICATION REPORT ===")
        f.write(report)
        f.write("=== CONFUSION MATRIX ===")
        f.write(cm_df.to_string())
        f.write("=== COMPACT METRICS ===")
        f.write(pd.DataFrame([metrics]).to_string(index=False))

    return pred_df, cm_df, confused_pairs, errors, metrics

print("Helpers defined.")


Helpers defined.


In [15]:
# =========================
# MASTER HELPER: TRAIN ONE MODEL END-TO-END
# =========================
def run_model(model_name, num_epochs=NUM_EPOCHS):
    print("\n" + "=" * 65)
    print(f"  MODEL: {model_name}")
    print("=" * 65)

    set_seed(RANDOM_STATE)
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)

    use_fast = False if "deberta-v3" in model_name else True
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=use_fast)

    train_ds, val_ds, test_ds = make_hf_datasets(
        tokenizer,
        X_train,
        y_train_ids,
        X_val,
        y_val_ids,
        X_test,
        y_test_ids,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(VALID_LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    short_name = model_name.split("/")[-1]
    output_dir = os.path.join(RESULTS_DIR, f"{short_name}_results")

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        num_train_epochs=num_epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
        fp16=torch.cuda.is_available() and "deberta-v3" not in model_name,
        save_total_limit=2,
    )

    trainer = WeightedTrainer(
        class_weights=class_weights_tensor,
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    print("Best validation macro-F1:", trainer.state.best_metric)

    _, _, _, _, train_metrics = evaluate_and_report(trainer, train_ds, y_train_ids, X_train, "TRAIN", short_name)
    _, _, _, _, val_metrics = evaluate_and_report(trainer, val_ds, y_val_ids, X_val, "VAL", short_name)
    _, _, _, _, test_metrics = evaluate_and_report(trainer, test_ds, y_test_ids, X_test, "TEST", short_name)

    all_metrics_df = pd.DataFrame([train_metrics, val_metrics, test_metrics])
    all_metrics_path = os.path.join(RESULTS_DIR, f"{short_name}_all_split_metrics.csv")
    all_metrics_df.to_csv(all_metrics_path, index=False)

    print("\n=== Compact metrics summary ===")
    display(all_metrics_df)
    print(f"Saved metrics summary: {all_metrics_path}")

    return trainer, test_metrics

print("run_model() defined.")


run_model() defined.


## Run Models

Run whichever models you need. BERT Base is listed first because it is the main comparison model.


In [16]:
bert_trainer, bert_metrics = run_model("bert-base-uncased", num_epochs=NUM_EPOCHS)



  MODEL: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

=== bert-base-uncased TEST TOP CONFUSED PAIRS ===


,true_label,predicted_label,count
0,Reveal,Interpret,31
1,Interpret,Reveal,16
2,Interpret,Revise,15
3,Chitchat,Interpret,12
4,Reveal,Chitchat,7
5,Chitchat,Reveal,7
6,Revise,Interpret,5
7,Interpret,Chitchat,5
8,Revise,Reveal,4
9,Inquire,Interpret,3



=== Compact metrics summary ===


,split,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,samples,chitchat_f1,chitchat_support,interpret_f1,interpret_support,reveal_f1,reveal_support,revise_f1,revise_support,inquire_f1,inquire_support
0,TRAIN,bert-base-uncased,0.901158,0.848512,0.840074,0.843415,0.902832,0.901158,0.901393,5352,0.834425,543,0.930249,2721,0.914573,1645,0.750330,360,0.7875,83
1,VAL,bert-base-uncased,0.837165,0.536945,0.534969,0.534724,0.833251,0.837165,0.833722,522,0.631579,36,0.877193,304,0.849057,168,0.315789,10,0.0000,4
2,TEST,bert-base-uncased,0.828221,0.522269,0.512909,0.515819,0.832583,0.828221,0.829758,652,0.743802,64,0.882812,378,0.819149,193,0.133333,12,0.0000,5


Saved metrics summary: /content/drive/MyDrive/intent9Aug/5classFinalscript_noleak_fullaugment_cn/bert-base-uncased_all_split_metrics.csv


In [17]:
distilbert_trainer, distilbert_metrics = run_model("distilbert-base-uncased", num_epochs=NUM_EPOCHS)



  MODEL: distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

=== distilbert-base-uncased TEST TOP CONFUSED PAIRS ===


,true_label,predicted_label,count
0,Interpret,Reveal,30
1,Reveal,Interpret,29
2,Chitchat,Interpret,19
3,Interpret,Revise,17
4,Revise,Interpret,7
5,Chitchat,Reveal,6
6,Revise,Reveal,4
7,Inquire,Interpret,3
8,Reveal,Revise,3
9,Inquire,Reveal,2



=== Compact metrics summary ===


,split,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,samples,chitchat_f1,chitchat_support,interpret_f1,interpret_support,reveal_f1,reveal_support,revise_f1,revise_support,inquire_f1,inquire_support
0,TRAIN,distilbert-base-uncased,0.931054,0.897723,0.925594,0.909025,0.932203,0.931054,0.930999,5352,0.888672,543,0.946735,2721,0.933742,1645,0.868874,360,0.907104,83
1,VAL,distilbert-base-uncased,0.825670,0.534791,0.536984,0.534674,0.822000,0.825670,0.823568,522,0.724638,36,0.865041,304,0.822823,168,0.260870,10,0.000000,4
2,TEST,distilbert-base-uncased,0.811350,0.528080,0.477912,0.494438,0.821497,0.811350,0.812475,652,0.742857,64,0.861619,378,0.807107,193,0.060606,12,0.000000,5


Saved metrics summary: /content/drive/MyDrive/intent9Aug/5classFinalscript_noleak_fullaugment_cn/distilbert-base-uncased_all_split_metrics.csv


In [18]:
roberta_trainer, roberta_metrics = run_model("roberta-base", num_epochs=NUM_EPOCHS)



  MODEL: roberta-base


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

=== roberta-base TEST TOP CONFUSED PAIRS ===


,true_label,predicted_label,count
0,Reveal,Interpret,33
1,Interpret,Reveal,21
2,Chitchat,Interpret,17
3,Interpret,Revise,12
4,Chitchat,Reveal,11
5,Revise,Interpret,9
6,Reveal,Revise,3
7,Inquire,Interpret,3
8,Revise,Reveal,3
9,Inquire,Reveal,2



=== Compact metrics summary ===


,split,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,samples,chitchat_f1,chitchat_support,interpret_f1,interpret_support,reveal_f1,reveal_support,revise_f1,revise_support,inquire_f1,inquire_support
0,TRAIN,roberta-base,0.913117,0.889504,0.893669,0.888559,0.915015,0.913117,0.912688,5352,0.852792,543,0.932109,2721,0.919889,1645,0.822751,360,0.915254,83
1,VAL,roberta-base,0.840996,0.567795,0.530854,0.546944,0.833186,0.840996,0.835266,522,0.707692,36,0.881517,304,0.829721,168,0.315789,10,0.000000,4
2,TEST,roberta-base,0.820552,0.515656,0.455640,0.476687,0.820875,0.820552,0.815813,652,0.699029,64,0.876117,378,0.808290,193,0.000000,12,0.000000,5


Saved metrics summary: /content/drive/MyDrive/intent9Aug/5classFinalscript_noleak_fullaugment_cn/roberta-base_all_split_metrics.csv


In [19]:
minilm_trainer, minilm_metrics = run_model("microsoft/MiniLM-L12-H384-uncased", num_epochs=NUM_EPOCHS)



  MODEL: microsoft/MiniLM-L12-H384-uncased


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

=== MiniLM-L12-H384-uncased TEST TOP CONFUSED PAIRS ===


,true_label,predicted_label,count
0,Reveal,Interpret,37
1,Chitchat,Interpret,21
2,Interpret,Revise,15
3,Interpret,Reveal,10
4,Revise,Interpret,8
5,Chitchat,Reveal,6
6,Reveal,Revise,5
7,Chitchat,Revise,3
8,Inquire,Interpret,3
9,Revise,Reveal,3



=== Compact metrics summary ===


,split,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,samples,chitchat_f1,chitchat_support,interpret_f1,interpret_support,reveal_f1,reveal_support,revise_f1,revise_support,inquire_f1,inquire_support
0,TRAIN,MiniLM-L12-H384-uncased,0.849028,0.655444,0.585641,0.612273,0.840366,0.849028,0.838191,5352,0.775465,543,0.883879,2721,0.898955,1645,0.503067,360,0.0,83
1,VAL,MiniLM-L12-H384-uncased,0.842912,0.550857,0.509507,0.522488,0.853292,0.842912,0.841997,522,0.730159,36,0.885449,304,0.848684,168,0.148148,10,0.0,4
2,TEST,MiniLM-L12-H384-uncased,0.826687,0.551213,0.466166,0.491866,0.843758,0.826687,0.825686,652,0.693878,64,0.882500,378,0.827397,193,0.055556,12,0.0,5


Saved metrics summary: /content/drive/MyDrive/intent9Aug/5classFinalscript_noleak_fullaugment_cn/MiniLM-L12-H384-uncased_all_split_metrics.csv


In [20]:
# =========================
# FINAL MODEL COMPARISON TABLE
# =========================
metrics_available = []
for name in ["bert_metrics", "distilbert_metrics", "roberta_metrics", "minilm_metrics"]:
    if name in globals():
        metrics_available.append(globals()[name])

comparison_df = pd.DataFrame(metrics_available)
if len(comparison_df):
    display_cols = [
        "model", "macro_f1", "accuracy", "weighted_f1",
        "chitchat_f1", "interpret_f1", "reveal_f1", "revise_f1", "inquire_f1",
        "samples",
    ]
    comparison_df = comparison_df[display_cols].sort_values("macro_f1", ascending=False).reset_index(drop=True)
    display(comparison_df)
    comparison_df.to_csv(os.path.join(RESULTS_DIR, "final_5class_model_comparison.csv"), index=False)
    print("Saved:", os.path.join(RESULTS_DIR, "final_5class_model_comparison.csv"))
else:
    print("No model metrics found yet. Run at least one model cell first.")


,model,macro_f1,accuracy,weighted_f1,chitchat_f1,interpret_f1,reveal_f1,revise_f1,inquire_f1,samples
0,bert-base-uncased,0.515819,0.828221,0.829758,0.743802,0.882812,0.819149,0.133333,0.0,652
1,distilbert-base-uncased,0.494438,0.811350,0.812475,0.742857,0.861619,0.807107,0.060606,0.0,652
2,MiniLM-L12-H384-uncased,0.491866,0.826687,0.825686,0.693878,0.882500,0.827397,0.055556,0.0,652
3,roberta-base,0.476687,0.820552,0.815813,0.699029,0.876117,0.808290,0.000000,0.0,652


Saved: /content/drive/MyDrive/intent9Aug/5classFinalscript_noleak_fullaugment_cn/final_5class_model_comparison.csv
